[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_02/10_snell_fresnel_y_brewster.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 10 — Snell, Fresnel y el ángulo de Brewster

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 2**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Calcular cuánta potencia se refleja y cuánta se transmite en incidencia
   normal, y comprobar que suman uno.
2. Aplicar la ley de Snell para obtener el ángulo de refracción.
3. Encontrar el ángulo crítico y explicar la reflexión interna total.
4. Encontrar el ángulo de Brewster y explicar por qué a ese ángulo una de las
   polarizaciones desaparece de la reflexión.

In [ ]:
# Preparación del entorno: local o Google Colab, con verificación SHA256.
import hashlib
import sys
import urllib.request
from pathlib import Path

MODULOS = {
    "utilidades_notebook.py": "e1811892d086ca99e03694c0e70853886f63be58326e3e7232c6978db1fdf7a9",
    "interfaces_planas.py": "6a244a9913aaa9ac0b0b220c717415bbfdb814add978ace5783279e61ecc9c02",
}
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


candidatos = [Path.cwd(), *Path.cwd().parents]
raiz_repo = next((p for p in candidatos if (p / ".git").exists()), None)
if raiz_repo is not None:
    for modulo, esperado in MODULOS.items():
        archivo = raiz_repo / "src" / modulo
        if not archivo.exists() or sha256(archivo) != esperado:
            raise RuntimeError(
                f"Hash local desactualizado para {modulo}. "
                "Ejecute scripts/refresh_notebook_hashes.py."
            )
    raiz = raiz_repo
else:
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo, esperado in MODULOS.items():
        destino = raiz / "src" / modulo
        if destino.exists() and sha256(destino) == esperado:
            continue
        with urllib.request.urlopen(URL_SRC + modulo, timeout=30) as respuesta:
            datos = respuesta.read()
        obtenido = hashlib.sha256(datos).hexdigest()
        if obtenido != esperado:
            raise RuntimeError(
                f"SHA256 inválido para {modulo}: {obtenido} != {esperado}"
            )
        destino.write_bytes(datos)

sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from interfaces_planas import (
    coeficiente_reflexion_normal,
    coeficiente_transmision_normal,
    reflectancia_normal,
    transmitancia_normal,
    coeficientes_fresnel,
    angulo_transmitido,
    angulo_critico,
    angulo_brewster,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 Todo sale de las condiciones de borde

Puede parecer que Fresnel y Snell son temas nuevos, pero no lo son: salen de
las mismas condiciones de borde de la semana 4. Exigir que las componentes
tangenciales de $\mathbf{E}$ y $\mathbf{H}$ sean continuas en la interfaz
obliga a que se cumplan tres cosas a la vez:

1. Las tres ondas —incidente, reflejada y transmitida— deben tener la misma
   frecuencia.
2. Sus fases deben coincidir a lo largo de la interfaz, y de ahí sale la ley
   de Snell.
3. Las amplitudes quedan determinadas, y de ahí salen los coeficientes de
   Fresnel.

### 2.2 Por qué hay dos polarizaciones

En incidencia oblicua, la interfaz rompe la simetría: ya no da lo mismo que
el campo esté en el plano de incidencia o perpendicular a él. Por eso hay dos
coeficientes distintos, $r_\perp$ y $r_\parallel$, y hay que tratarlos por
separado.

En incidencia normal esa distinción desaparece, y por eso allí basta un solo
coeficiente $\Gamma$.

### 2.3 Dos ángulos especiales

- **Ángulo crítico.** En dos dieléctricos ideales, solo existe al pasar
  del medio de mayor índice al de menor índice. Más allá aparece una onda
  evanescente en el segundo medio, pero no hay potencia neta transmitida en
  la dirección normal: ocurre reflexión interna total. Este confinamiento es
  la base del modelo ideal de una fibra óptica; una fibra real también tiene
  absorción, dispersión y pérdidas por curvatura.
- **Ángulo de Brewster.** Para dieléctricos no magnéticos, sin pérdidas y
  con índices distintos, existe un ángulo en el cual la polarización
  paralela no se refleja. La luz reflejada queda entonces polarizada en la
  dirección perpendicular. En materiales con pérdidas no necesariamente hay
  un cero exacto de reflexión.

## 3. Ecuaciones

**Incidencia normal:**

$$
\Gamma = \frac{n_1 - n_2}{n_1 + n_2},
\qquad
\tau = 1 + \Gamma,
$$

$$
R = |\Gamma|^{2},
\qquad
T = \frac{n_2}{n_1}|\tau|^{2},
\qquad
R + T = 1 .
$$

El factor $n_2/n_1$ en $T$ no es un capricho: hace falta porque la potencia
depende del medio, no solo de la amplitud del campo.

**Ley de Snell:**

$$
n_1\sin\theta_i = n_2\sin\theta_t .
$$

**Coeficientes de Fresnel** para las dos polarizaciones:

$$
r_\perp = \frac{n_1\cos\theta_i - n_2\cos\theta_t}{n_1\cos\theta_i + n_2\cos\theta_t},
\qquad
r_\parallel = \frac{n_2\cos\theta_i - n_1\cos\theta_t}{n_2\cos\theta_i + n_1\cos\theta_t}.
$$

**Ángulos especiales:**

$$
\sin\theta_c = \frac{n_2}{n_1}\quad (\text{requiere } n_1 > n_2),
\qquad
\tan\theta_B = \frac{n_2}{n_1}.
$$

## 4. Qué significa físicamente

**Del aire al vidrio se refleja poco.** Con $n_1 = 1.0$ y $n_2 = 1.5$ resulta
$R = 0.04$: solo el 4 % de la potencia rebota. Por eso a través de una
ventana se ve bien, aunque de noche también se vea su reflejo.

**$\Gamma$ es negativo, y eso significa inversión.** El signo menos indica
que la onda reflejada sale invertida respecto de la incidente. Es lo mismo
que ocurre con un pulso en una cuerda cuando llega a un extremo fijo.

**La reflexión interna total es realmente total.** Más allá del ángulo
crítico, $\cos\theta_t$ se vuelve imaginario, $|r| = 1$ exactamente, y no se
pierde nada de potencia. Por eso la luz puede viajar kilómetros dentro de una
fibra óptica.

**En Brewster desaparece una polarización, no la reflexión.** A ese ángulo
$r_\parallel = 0$, pero $r_\perp$ sigue siendo grande. La luz reflejada no
desaparece: queda polarizada. Un lente de sol polarizado se orienta para
bloquear precisamente esa componente, y así elimina el brillo del agua o del
asfalto.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: incidencia normal ---
n1 = 1.0                 # índice de refracción del medio de entrada (aire)
n2 = 1.5                 # índice de refracción del medio de salida (vidrio)
campo_incidente = 10.0   # amplitud del campo incidente [V/m]

# --- Problema 2: incidencia oblicua ---
angulo_incidencia_grados = 55.0  # ángulo de incidencia [grados]

## 6. Implementación

### 6.1 Problema 1 — incidencia normal

In [ ]:
Gamma = coeficiente_reflexion_normal(n1, n2)
tau = coeficiente_transmision_normal(n1, n2)

campo_reflejado = Gamma * campo_incidente
campo_transmitido = tau * campo_incidente

R = reflectancia_normal(n1, n2)
T = transmitancia_normal(n1, n2)

### 6.2 Problema 2 — incidencia oblicua y ángulos especiales

In [ ]:
angulo_incidencia = np.deg2rad(angulo_incidencia_grados)

# El ángulo crítico solo existe al salir del medio denso hacia el menos denso.
theta_critico = np.rad2deg(angulo_critico(n2, n1))
theta_brewster = np.rad2deg(angulo_brewster(n1, n2))

r_perpendicular, r_paralelo, coseno_t = coeficientes_fresnel(n1, n2, angulo_incidencia)
seno_t = n1 / n2 * np.sin(angulo_incidencia)
hay_rayo_transmitido = abs(seno_t) <= 1.0
theta_transmitido = (
    np.rad2deg(float(np.real(angulo_transmitido(n1, n2, angulo_incidencia))))
    if hay_rayo_transmitido
    else np.nan
)

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Coeficiente de reflexión", "Gamma", Gamma, "-"),
        ("Coeficiente de transmisión", "tau", tau, "-"),
        ("Campo reflejado", "E_r", campo_reflejado, "V/m"),
        ("Campo transmitido", "E_t", campo_transmitido, "V/m"),
        ("Potencia reflejada", "R", R, "-"),
        ("Potencia transmitida", "T", T, "-"),
        ("Suma (debe dar 1)", "R + T", R + T, "-"),
    ]
)

In [ ]:
tabla_resultados(
    [
        (f"Ángulo crítico (de n={n2} a n={n1})", "theta_c", theta_critico, "grados"),
        (f"Ángulo de Brewster (de n={n1} a n={n2})", "theta_B", theta_brewster, "grados"),
        ("Ángulo transmitido", "theta_t", theta_transmitido, "grados"),
        ("Reflectancia perpendicular", "R_perp", abs(r_perpendicular) ** 2, "-"),
        ("Reflectancia paralela", "R_par", abs(r_paralelo) ** 2, "-"),
    ]
)

In [ ]:
print(f"Conservación de la potencia: R + T = {R + T:.12f}")
assert np.isclose(R + T, 1.0), "La potencia no se conserva"
print("La potencia se conserva.")

## 8. Visualización

Reflectancia de cada polarización en función del ángulo. A la izquierda,
entrando al medio denso; a la derecha, saliendo de él.

In [ ]:
angulos = np.deg2rad(np.linspace(0.0, 89.9, 800))
configuraciones = [
    ((n1, n2), f"De n = {n1} a n = {n2}"),
    ((n2, n1), f"De n = {n2} a n = {n1}"),
]

fig, ejes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=True)
for eje, (par, titulo) in zip(ejes, configuraciones):
    rs, rp, _ = coeficientes_fresnel(par[0], par[1], angulos)
    eje.plot(np.rad2deg(angulos), np.abs(rs) ** 2, label="perpendicular")
    eje.plot(np.rad2deg(angulos), np.abs(rp) ** 2, label="paralela")
    if par[0] > par[1]:
        eje.axvline(np.rad2deg(angulo_critico(par[0], par[1])), color="black",
                    linestyle="--", label="ángulo crítico")
    eje.axvline(np.rad2deg(angulo_brewster(par[0], par[1])), color="gray",
                linestyle=":", label="Brewster")
    eje.set_xlabel("Ángulo de incidencia (grados)")
    eje.set_title(titulo)
    eje.legend(fontsize=8)
ejes[0].set_ylabel("Fracción de potencia reflejada")
fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**En incidencia normal se transmite el 96 %.** El valor de $R + T$ coincide
con 1 dentro de la tolerancia numérica. El `assert` verifica ese balance para
los medios sin pérdidas del modelo.

**La curva paralela toca el cero.** En el panel izquierdo, la curva de la
polarización paralela baja hasta cero justo en la línea punteada de Brewster,
a 56.3°. La perpendicular nunca se anula. Ésa es toda la física de un lente
polarizado.

**A 55° ya casi estamos en Brewster.** La reflectancia paralela es del orden
de $10^{-4}$: prácticamente nada. La perpendicular, en cambio, es de un 14 %.
La luz reflejada a ese ángulo está casi totalmente polarizada.

**A la derecha aparece un muro vertical.** Al salir del vidrio, las dos curvas
suben a 1 en el ángulo crítico (41.8°) y se quedan allí. Ese salto es la
reflexión interna total: la fibra óptica funciona manteniendo la luz siempre
más allá de ese ángulo.

## 10. Ejercicios para experimentar

            1. Cambie `n2` a `1.33` (agua). ¿Cuánto se refleja ahora en incidencia
               normal? ¿Dónde queda el ángulo de Brewster?
            2. Ponga `n2 = 1.0`, igual a `n1`. ¿Cuánto valen $\Gamma$ y $R$? ¿Tiene
               sentido? ¿Hay interfaz?
            3. Ponga `n2 = 4.0` (silicio en el infrarrojo). ¿Cuánta potencia se pierde por
               reflexión? Ese es el problema que resuelven las capas antirreflectantes de
               las celdas solares.
            4. Cambie `angulo_incidencia_grados` al valor de Brewster que aparece en la
               tabla. ¿Cuánto vale $R_{\parallel}$?
            5. Ponga `angulo_incidencia_grados = 80.0`. ¿Qué pasa con ambas
               reflectancias cerca de la incidencia rasante? ¿Por qué el agua de un lago
               refleja como espejo cuando uno la mira de lejos?
            6. Intercambie `n1` y `n2` y ponga `angulo_incidencia_grados = 50.0`.
               Compare con el ángulo crítico de la tabla: ¿hay un rayo transmitido
               propagante? La tabla debe mostrar `nan` para su ángulo si hay reflexión interna total.